# Day 033 — Exercise 3: throttled_gather

**What you'll build:** `throttled_gather(coros, max_concurrent) -> list` — run coroutines concurrently with `asyncio.Semaphore` capping simultaneous execution to at most `max_concurrent`.

**Why it matters:** `asyncio.gather` with 1000 coroutines starts 1000 simultaneous requests. A local Ollama server can only handle a few at a time. `throttled_gather` prevents server overload while keeping concurrent throughput.

## Provided: gather_results

In [ ]:
import asyncio

async def gather_results(coros: list) -> list:
    return list(await asyncio.gather(*coros))

## Your Implementation

In [ ]:
async def throttled_gather(coros: list, max_concurrent: int) -> list:
    """
    Run coros concurrently, at most max_concurrent at a time.

    Args:
        coros:          List of coroutine objects.
        max_concurrent: Maximum number of coroutines running simultaneously.

    Returns:
        List of results in input order.
    """
    # TODO: sem = asyncio.Semaphore(max_concurrent)
    # TODO: async def _run(coro):
    #     async with sem:
    #         return await coro
    # TODO: return list(await asyncio.gather(*[_run(c) for c in coros]))
    pass

## Check Your Work

In [ ]:
import asyncio
import time

async def _run_checks():
    total = 5
    passed = 0

    async def _slow(x, delay: float = 0.05):
        await asyncio.sleep(delay)
        return x

    # Check 1: defined
    try:
        assert 'throttled_gather' in globals()
        passed += 1; print('\u2705 Check 1: throttled_gather defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    # Check 2: returns correct values
    try:
        result = await throttled_gather([_slow(10), _slow(20), _slow(30)], 2)
        assert result == [10, 20, 30], \
            f'expected [10, 20, 30], got {result}'
        passed += 1; print('\u2705 Check 2: correct results returned')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: order preserved
    try:
        result = await throttled_gather([_slow(99), _slow(1), _slow(50)], 2)
        assert result == [99, 1, 50], \
            f'order not preserved: {result}'
        passed += 1; print('\u2705 Check 3: input order preserved')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: max_concurrent=1 is serial — 3×0.05s tasks take ≥ 0.12s
    try:
        start = time.time()
        await throttled_gather([_slow(1), _slow(2), _slow(3)], max_concurrent=1)
        elapsed = time.time() - start
        assert elapsed >= 0.12, \
            f'max_concurrent=1 should be serial (≥0.12s), took {elapsed:.3f}s'
        passed += 1; print(f'\u2705 Check 4: max_concurrent=1 is serial ({elapsed:.3f}s ≥ 0.12s)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: max_concurrent=4 is concurrent — 4×0.05s tasks take < 0.15s
    try:
        start = time.time()
        await throttled_gather([_slow(i) for i in range(4)], max_concurrent=4)
        elapsed = time.time() - start
        assert elapsed < 0.15, \
            f'max_concurrent=4 should be concurrent (< 0.15s), took {elapsed:.3f}s'
        passed += 1; print(f'\u2705 Check 5: max_concurrent=4 is concurrent ({elapsed:.3f}s < 0.15s)')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


await _run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
async def throttled_gather(coros: list, max_concurrent: int) -> list:
    sem = asyncio.Semaphore(max_concurrent)
    async def _run(coro):
        async with sem:
            return await coro
    return list(await asyncio.gather(*[_run(c) for c in coros]))
```

</details>